# Phase 4 — Publication Histories, Treated Authors

Extracts the complete publication history of every author passing the Phase 3
screen.

**Input:** `data/interim/phase04_author_queue.csv`, the OpenAlex snapshot

**Outputs**

| File | Contents |
|---|---|
| `data/interim/phase04_papers.csv` | one row per author per paper |

One row is written per author-work pair. A paper co-authored by two queued
authors appears twice, once under each, because the panel is constructed at
author level.

Rows are written to disk as each batch is processed rather than accumulated in
memory. The output runs to several million rows, and the works carrying them
hold nested author and location structures that are an order of magnitude
larger than the fields retained.

Author names, affiliations and byline position on these papers are not
retained. Position on the retracted paper is recorded in Phase 2; position on
an author's other papers is not used by any hypothesis, and the `authorships`
field is the largest column in the source.

In [1]:
import csv
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

from snapshot import Snapshot, strip_doi, strip_id

QUEUE = "data/interim/phase04_author_queue.csv"
PAPER_LEVEL = "data/interim/phase02_paper_level.csv"

OUT_PAPERS = "data/interim/phase04_papers.csv"

PAPER_COLUMNS = [
    "author_id", "work_id", "doi", "pub_year", "type", "is_retracted_flag",
    "is_focal_retraction", "cited_by_count", "counts_by_year",
    "source_id", "source_name",
]

SCAN_COLUMNS = ["id", "doi", "publication_year", "type", "is_retracted",
                "cited_by_count", "counts_by_year", "authorships",
                "primary_location"]

# Restrict the scan for testing. None runs the full snapshot.
LIMIT_FILES = None
SKIP_FILES = 0

pd.set_option("display.width", 200)
os.makedirs("data/interim", exist_ok=True)

snap = Snapshot()
d = snap.describe("works")
print(f"works: {d['files']:,} files, {d['bytes'] / 2**30:.0f} GiB")

works: 2,446 files, 675 GiB


## The queue

In [2]:
queue = pd.read_csv(QUEUE, low_memory=False)
queue["author_id"] = queue.author_id.astype(str)
author_ids = set(queue.author_id)

print(f"authors queued  {len(author_ids):,}")
print(queue.first_category.value_counts().to_string())

focal_ids = set()
if os.path.isfile(PAPER_LEVEL):
    focal_ids = set(pd.read_csv(PAPER_LEVEL, usecols=["openalex_id"])
                      .openalex_id.astype(str))
    print(f"\nfocal retracted papers  {len(focal_ids):,}")
else:
    print(f"\n{PAPER_LEVEL} not found; is_focal_retraction will be False "
          f"throughout")

authors queued  55,821
first_category
AUTHOR_MISCONDUCT       29884
HONEST_ERROR            11962
EDITORIAL_COMPROMISE     8004
UNCONFIRMED_CONCERNS     4747
ETHICS_VIOLATION          822
UNCLASSIFIED              402

focal retracted papers  17,591


## Extraction

The `authorships` column is flattened once per batch and tested against the
queue in a single operation. Work-level fields are then taken at the matching
row positions, so a work with two queued authors yields two rows without the
work being read twice.

In [3]:
PREFIX = "https://openalex.org/"
want = pa.array(sorted(author_ids | {PREFIX + a for a in author_ids}),
                type=pa.string())


def compact_counts(rows):
    """counts_by_year as 'YYYY:n|YYYY:n', newest first."""
    out = []
    for r in rows:
        if not r:
            out.append("")
            continue
        pairs = sorted(((c["year"], c.get("cited_by_count", 0)) for c in r
                        if c.get("year") is not None),
                       key=lambda p: p[0], reverse=True)
        out.append("|".join(f"{y}:{n}" for y, n in pairs))
    return out


class Extractor:
    """Writes matching author-work rows as each batch is processed."""

    def __init__(self, path, columns):
        self.f = open(path, "w", newline="", encoding="utf-8")
        self.writer = csv.DictWriter(self.f, fieldnames=columns)
        self.writer.writeheader()
        self.n_rows = 0
        self.authors_seen = set()

    def handle(self, tbl, path):
        auth_col = tbl.column("authorships")
        if isinstance(auth_col, pa.ChunkedArray):
            auth_col = auth_col.combine_chunks()

        parent = pc.list_parent_indices(auth_col)
        ids = auth_col.values.field("author").field("id")

        hit = pc.is_in(ids, value_set=want)
        sel_ids = pc.filter(ids, hit)
        if len(sel_ids) == 0:
            return None
        sel_parent = pc.filter(parent, hit)

        # Work-level fields taken at the matching positions, so a work with
        # several queued authors is expanded rather than reread.
        w = tbl.take(sel_parent)
        loc = w.column("primary_location").to_pylist()

        rows = pd.DataFrame({
            "author_id": [strip_id(x) for x in sel_ids.to_pylist()],
            "work_id": [strip_id(x) for x in w.column("id").to_pylist()],
            "doi": [strip_doi(x) for x in w.column("doi").to_pylist()],
            "pub_year": w.column("publication_year").to_pylist(),
            "type": w.column("type").to_pylist(),
            "is_retracted_flag": [bool(x) for x in
                                  w.column("is_retracted").to_pylist()],
            "cited_by_count": w.column("cited_by_count").to_pylist(),
            "counts_by_year": compact_counts(
                w.column("counts_by_year").to_pylist()),
            # primary_location.id identifies the location record; the venue
            # is one level down at primary_location.source.id.
            "source_id": [strip_id(((p or {}).get("source") or {}).get("id"))
                          if p else None for p in loc],
            "source_name": [((p or {}).get("source") or {}).get("display_name")
                            if p else None for p in loc],
        })
        rows["is_focal_retraction"] = rows.work_id.isin(focal_ids)

        self.writer.writerows(rows[PAPER_COLUMNS].to_dict("records"))
        self.f.flush()
        self.n_rows += len(rows)
        self.authors_seen.update(rows.author_id)
        return None

    def close(self):
        self.f.close()


ex = Extractor(OUT_PAPERS, PAPER_COLUMNS)
t0 = time.time()
try:
    snap.scan("works", SCAN_COLUMNS, ex.handle,
              limit_files=LIMIT_FILES, skip_files=SKIP_FILES,
              progress_every=200)
finally:
    ex.close()

print(f"\nelapsed {(time.time() - t0) / 60:.1f} min")
print(f"rows written    {ex.n_rows:,}")
print(f"authors found   {len(ex.authors_seen):,} of {len(author_ids):,}")

scanning works: 2,446 files, 675.2 GiB on disk
  projecting 9 of 49 columns
  200/2,446 files | 0.0M rows | kept 0 | 16 MiB/s | eta 704m
  400/2,446 files | 46.7M rows | kept 0 | 193 MiB/s | eta 55m
  600/2,446 files | 81.3M rows | kept 0 | 176 MiB/s | eta 57m
  800/2,446 files | 119.0M rows | kept 0 | 170 MiB/s | eta 54m
  1,000/2,446 files | 155.3M rows | kept 0 | 165 MiB/s | eta 52m
  1,200/2,446 files | 192.2M rows | kept 0 | 161 MiB/s | eta 48m
  1,400/2,446 files | 230.6M rows | kept 0 | 159 MiB/s | eta 44m
  1,600/2,446 files | 271.4M rows | kept 0 | 150 MiB/s | eta 42m
  1,800/2,446 files | 326.2M rows | kept 0 | 146 MiB/s | eta 36m
  2,000/2,446 files | 380.3M rows | kept 0 | 145 MiB/s | eta 28m
  2,200/2,446 files | 434.0M rows | kept 0 | 125 MiB/s | eta 16m
  2,400/2,446 files | 493.9M rows | kept 0 | 119 MiB/s | eta 3m
  done: 510,372,821 rows scanned, 0 kept, 97.7 min

elapsed 97.7 min
rows written    7,806,704
authors found   55,821 of 55,821


## Coverage

An author in the queue with no rows here has no works in the snapshot under
that identifier. A small number is expected from entity merges between the
`authors` and `works` entities; a large number indicates an identifier
mismatch.

In [4]:
missing = author_ids - ex.authors_seen
print(f"authors with no works: {len(missing):,} "
      f"({len(missing) / len(author_ids):.2%})")

if missing:
    m = queue[queue.author_id.isin(missing)]
    print(f"\nby category")
    print(m.first_category.value_counts().to_string())
    print(f"\nworks_count reported by the authors entity")
    print(m.works_count.describe().round(1).to_string())

authors with no works: 0 (0.00%)


## The extract

In [5]:
papers = pd.read_csv(OUT_PAPERS, low_memory=False)
print(f"rows            {len(papers):,}")
print(f"authors         {papers.author_id.nunique():,}")
print(f"distinct works  {papers.work_id.nunique():,}")

dupes = int(papers.duplicated(subset=["author_id", "work_id"]).sum())
if dupes:
    print(f"\n  {dupes:,} duplicate author-work rows")

print(f"\npapers per author")
per = papers.groupby("author_id").size()
print(f"  median          {per.median():.0f}")
print(f"  mean            {per.mean():.1f}")
print(f"  90th percentile {per.quantile(0.9):.0f}")
print(f"  maximum         {per.max():,}")

print(f"\npublication years {int(papers.pub_year.min())} to "
      f"{int(papers.pub_year.max())}")
print(f"focal retractions  {int(papers.is_focal_retraction.sum()):,}")
print(f"OpenAlex retraction flag set on "
      f"{int(papers.is_retracted_flag.sum()):,} rows")

sid = papers.source_id.dropna().astype(str)
print(f"\nsource_id present on {papers.source_id.notna().mean():.1%} of rows")
print(f"  distinct sources  {sid.nunique():,}")
print(f"  examples          {sid.head(3).tolist()}")
if len(sid) and not sid.str.match(r"^S\d+$").all():
    bad = sid[~sid.str.match(r"^S\d+$")].head(3).tolist()
    print(f"  [!] not all source ids are OpenAlex source identifiers: {bad}")

odd = papers[(papers.pub_year < 1900) | (papers.pub_year > 2027)]
print(f"\nrows outside 1900-2027  {len(odd):,}  "
      f"({len(odd) / len(papers):.3%})")
if len(odd):
    print("  " + odd.pub_year.value_counts().head(6).to_string()
                    .replace("\n", "\n  "))

rows            7,806,704
authors         55,821
distinct works  6,274,801

  31,257 duplicate author-work rows

papers per author
  median          67
  mean            139.9
  90th percentile 326
  maximum         10,898

publication years 1556 to 2029
focal retractions  70,501
OpenAlex retraction flag set on 113,340 rows

source_id present on 90.2% of rows
  distinct sources  91,877
  examples          ['S4306401568', 'S4306401568', 'S4306401568']

rows outside 1900-2027  1,080  (0.014%)
  pub_year
  1890.0    120
  1880.0     73
  1884.0     49
  1883.0     47
  1894.0     42
  1891.0     41


## Coverage of the analysis window

The panel requires six years either side of a retraction. This reports how many
authors have publication activity across that span, which bounds what the
balanced-window filter at Phase 8 can retain.

In [6]:
q = queue.set_index("author_id")
p = papers.dropna(subset=["pub_year"]).copy()
p["pub_year"] = p.pub_year.astype(int)
p["retraction_year"] = p.author_id.astype(str).map(q.first_retraction_year)
p = p.dropna(subset=["retraction_year"])
p["event_time"] = p.pub_year - p.retraction_year.astype(int)

span = p.groupby("author_id").event_time.agg(["min", "max"])
for pre, post in [(3, 3), (6, 6)]:
    ok = int(((span["min"] <= -pre) & (span["max"] >= post)).sum())
    print(f"activity spanning -{pre} to +{post}:  {ok:,} authors "
          f"({ok / len(span):.1%})")

print(f"\nrows by event time, -8 to +8")
et = p[(p.event_time >= -8) & (p.event_time <= 8)]
print(et.event_time.value_counts().sort_index().to_string())

span2 = span.join(q[["first_category"]])
ok = span2[(span2["min"] <= -6) & (span2["max"] >= 6)]
tab = pd.DataFrame({
    "queued": q.first_category.value_counts(),
    "spanning": ok.first_category.value_counts(),
}).fillna(0).astype(int)
tab["rate"] = (100 * tab.spanning / tab.queued).round(1)
print(f"\nby category, activity spanning -6 to +6")
print(tab.to_string())

activity spanning -3 to +3:  48,228 authors (86.4%)
activity spanning -6 to +6:  20,618 authors (36.9%)

rows by event time, -8 to +8
event_time
-8    243745
-7    266115
-6    295201
-5    310665
-4    340791
-3    370832
-2    411852
-1    453187
 0    497944
 1    475217
 2    482275
 3    478256
 4    383635
 5    282413
 6    207471
 7    156586
 8    113364

by category, activity spanning -6 to +6
                      queued  spanning  rate
first_category                              
AUTHOR_MISCONDUCT      29884     10522  35.2
HONEST_ERROR           11962      5206  43.5
EDITORIAL_COMPROMISE    8004      2793  34.9
UNCONFIRMED_CONCERNS    4747      1566  33.0
ETHICS_VIOLATION         822       344  41.8
UNCLASSIFIED             402       187  46.5


## Citation coverage

`counts_by_year` supports the citation outcome. Records without it contribute
zeros, so the share carrying it bounds what that outcome can be estimated on.

In [7]:
has_counts = papers.counts_by_year.notna() & (papers.counts_by_year != "")
print(f"rows with counts_by_year: {int(has_counts.sum()):,} "
      f"({has_counts.mean():.1%})")

with_counts = papers[has_counts]
if len(with_counts):
    years = (with_counts.counts_by_year.str.split("|").str[0]
                        .str.split(":").str[0])
    print(f"\nmost recent year present, ten most frequent")
    print(years.value_counts().head(10).to_string())

rows with counts_by_year: 5,312,686 (68.1%)

most recent year present, ten most frequent
counts_by_year
2026    2433611
2025    1274356
2024     490690
2023     281396
2022     191428
2021     156095
2020     108595
2019      78643
2018      62752
2017      52955
